# Fase 7 — Demo do assistente

**Objetivo da fase:** carregar o modelo base com o adaptador LoRA salvo no Drive e colocá-lo atrás de uma demo no Gradio, onde o analista cola um alerta com o contexto do enriquecimento e recebe a triagem em JSON.

## 7.1 Preparar o ambiente

### 7.1.1 Configurar o Google Colab

Em *Ambiente de execução → Alterar o tipo de ambiente de execução*, selecionar **GPU T4** e **Versão 2026.10**, a mesma da Fase 1. Usar um **runtime novo**, sem nada da Fase 6 na memória: o adaptador vem do Drive e o modelo base vem do Hub. Isso confere que a demo funciona só com o que foi salvo, como aconteceria num ambiente novo do SOC.

O secret `HF_TOKEN` criado na Fase 1 já fica disponível para todos os notebooks da conta. No painel lateral, ícone de chave, ligar *Acesso ao notebook* para o `HF_TOKEN` e rodar a célula abaixo para verificar a configuração.

In [2]:
from google.colab import userdata
from huggingface_hub import login, whoami

login(token=userdata.get("HF_TOKEN"))

conta = whoami()
print(f"Conta:         {conta['name']}")
print(f"Tipo de token: {conta['auth']['accessToken']['role']}")
# Esperado: o nome da conta e "write"

Conta:         poliana-moraes
Tipo de token: write


### 7.1.2 Instalar as bibliotecas

Instalar as bibliotecas através das linhas de comando abaixo. O `peft` carrega o adaptador sobre o modelo base e o `gradio` monta a interface web:

In [3]:
# Versões fixadas para o runtime 2026.10 do Colab (ver README)
!pip install -q transformers==5.22.0 datasets==4.8.5 peft==0.21.2 accelerate==1.15.0 huggingface_hub==1.33.0 gradio==6.31.0
# O runtime do Colab traz o torchao 0.10.0, que o peft recusa ao ser importado. O projeto não usa o torchao (não há quantização)
!pip uninstall -q -y torchao

## 7.2 Carregar o assistente

### 7.2.1 Carregar o adaptador do Drive

Montar o Google Drive, ler a configuração da Fase 4 e carregar o modelo base com o adaptador, do mesmo jeito que na seção 6.2.1. É o adaptador treinado na Fase 5 e medido na Fase 6.

Ao rodar a célula, o Colab pede permissão para acessar o Google Drive da conta:

In [4]:
import json
import torch
from google.colab import drive
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

drive.mount("/content/drive")

PROJETO_DIR = "/content/drive/MyDrive/fine-tuning-soc"
ADAPTER_DIR = f"{PROJETO_DIR}/soc-triage-qwen2.5-0.5b-lora"

with open(f"{PROJETO_DIR}/config_treino.json") as f:
    config_treino = json.load(f)

MODEL_ID = config_treino["model_id"]

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()

print(f"Modelo base: {MODEL_ID}")
print(f"Adaptador:   {ADAPTER_DIR}")
print(f"Memória da GPU: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
# Esperado: cerca de 1 GB, o modelo base em fp16 mais os poucos MB do adaptador

Mounted at /content/drive


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo base: Qwen/Qwen2.5-0.5B-Instruct
Adaptador:   /content/drive/MyDrive/fine-tuning-soc/soc-triage-qwen2.5-0.5b-lora
Memória da GPU: 1.00 GB


Mostrar as métricas salvas na Fase 6, para que quem usar a demo saiba com que resultado o assistente foi medido no teste:

In [6]:
with open(f"{PROJETO_DIR}/avaliacao.json") as f:
    ft = json.load(f)["metricas"]["Fine-tuned"]

print(f"JSON válido {ft['JSON válido']:.0%} | veredito {ft['Acurácia de veredito']:.0%} | "
      f"recall de crítica {ft['Recall de crítica']:.0%}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/fine-tuning-soc/avaliacao.json'

### 7.2.2 Carregar o prompt de sistema e os exemplos

A demo precisa do **mesmo `SYSTEM`** usado no treino e na avaliação, a regra de ouro da seção 3.4.1. Em vez de copiá-lo à mão, ele é lido do próprio dataset da Fase 3, então não há risco de uma vírgula diferente.

O split de **teste** da Fase 2 também fornece alertas prontos para os exemplos da demo, com o `alerta` e o `contexto` separados:

In [7]:
from datasets import load_dataset

SPLITS_ID = f"{conta['name']}/alertas-soc-splits"

teste_sft = load_dataset(config_treino["sft_id"], split="test")
teste = load_dataset(SPLITS_ID, split="test")

SYSTEM = teste_sft[0]["prompt"][0]["content"]
assert all(ex["prompt"][0]["content"] == SYSTEM for ex in teste_sft), "o SYSTEM varia entre os exemplos"
print(SYSTEM)
# Esperado: o mesmo SYSTEM da seção 3.3

README.md:   0%|          | 0.00/662 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  686kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  152kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  145kB            

... (11 outputs left out)

Generating train split:   0%|          | 0/1033 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/224 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/221 [00:00<?, ? examples/s]

Você é um analista de SOC. Aplique a política de severidade da organização e responda apenas com o JSON da triagem.


## 7.3 Função de triagem

### 7.3.1 Gerar e validar

A função recebe o alerta e o contexto, monta o prompt **no mesmo formato do treino** (`SYSTEM` + alerta e contexto separados por quebra de linha) e gera com `do_sample=False`, como na Fase 6: o mesmo alerta sempre recebe a mesma triagem.

A saída passa pela **mesma validação estrita** da seção 6.4.1. Numa integração real, uma saída inválida não pode seguir para a fila do SOC nem para o SOAR, então a demo também a trata como falha, e não tenta "consertar" o JSON:

In [8]:
VEREDITOS = {"falso_positivo", "verdadeiro_positivo"}
SEVERIDADES = {"baixa", "media", "alta", "critica"}
CHAVES = {"veredito", "severidade", "justificativa"}


def validar(saida: str) -> dict | None:
    try:
        d = json.loads(saida)
    except json.JSONDecodeError:
        return None
    if not isinstance(d, dict) or set(d) != CHAVES:
        return None
    if d["veredito"] not in VEREDITOS or d["severidade"] not in SEVERIDADES:
        return None
    return d


def triar(alerta: str, contexto: str) -> tuple[str, dict | None]:
    prompt = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"{alerta.strip()}\n{contexto.strip()}"},
    ]
    inputs = tokenizer.apply_chat_template(
        prompt, add_generation_prompt=True, return_dict=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    saida = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return saida, validar(saida)

### 7.3.2 Conferir antes de abrir a demo

Rodar a função num alerta de cada severidade do teste e medir o tempo de cada triagem. O prompt é montado pela função, e não lido do dataset da Fase 3, então a comparação com a triagem esperada também confere que o formato da demo é igual ao do treino:

In [9]:
import time

for severidade in ["critica", "alta", "media", "baixa"]:
    ex = next(ex for ex in teste if ex["triagem"]["severidade"] == severidade)
    inicio = time.perf_counter()
    saida, triagem = triar(ex["alerta"], ex["contexto"])
    tempo = time.perf_counter() - inicio

    print(f"=== {severidade} ({tempo:.1f} s) ===")
    print(ex["alerta"])
    print(ex["contexto"])
    print(f"Esperado: {json.dumps(ex['triagem'], ensure_ascii=False)}")
    print(f"Previsto: {saida if triagem else 'INVÁLIDA → ' + saida}\n")
# Esperado: 4 saídas em JSON válido, cada uma em poucos segundos na T4

=== critica (4.8 s) ===
EDR | Host: SRV-PAG-08 | Usuário: j.silva | Processo: rundll32.exe upd.dll,Start | Pai: outlook.exe
Ativo: servidor de pagamentos (criticidade: crítico) | Usuário: assistente administrativo, sem privilégio admin | Reputação do hash: maliciosa | Ocorrências em 7 dias: 1
Esperado: {"veredito": "verdadeiro_positivo", "severidade": "critica", "justificativa": "DLL com reputação maliciosa carregada via rundll32 a partir do Outlook, e ocorreu em servidor de pagamentos, ativo crítico."}
Previsto: {"veredito": "verdadeiro_positivo", "severidade": "critica", "justificativa": "upd.dll é redirecionado pelo Outlook, e hash maliciosa foi usada; o ativo, servidor de pagamentos, é um dos mais sensíveis da rede."}

=== alta (3.8 s) ===
EDR | Host: SRV-PRINT-04 | Usuário: m.souza | Processo: rundll32.exe cache32.dll,Start | Pai: outlook.exe
Ativo: servidor de impressão (criticidade: comum) | Usuário: analista de compras, sem privilégio admin | Reputação do hash: maliciosa | Ocor

**Leitura:** o tempo por alerta é o argumento de **volume e custo** do README na prática. Um analista L1 leva minutos para ler o alerta, consultar o contexto e aplicar a política. O modelo leva segundos numa GPU de entrada, sem enviar nada para fora do ambiente.

Se alguma triagem divergir da esperada, isso já era esperado pelos números da Fase 6: a demo mostra o modelo como ele é, com os mesmos acertos e erros medidos no teste.

## 7.4 Demo no Gradio

### 7.4.1 Montar a interface

A interface segue as etapas do SOC descritas no README:

- **Alerta:** a linha que o SIEM entrega depois da normalização (etapa 2).
- **Contexto:** o resultado do enriquecimento, com CMDB, IAM, Threat Intel e histórico (etapa 3).
- **Triagem:** o JSON que o modelo devolve (etapa 4), pronto para a fila do L1 ou para o playbook do SOAR (etapa 5).

Os exemplos vêm do conjunto de teste, um por severidade e um falso positivo, para que quem usar a demo comece por alertas que o modelo nunca viu no treino:

In [10]:
import gradio as gr

ICONES = {"critica": "🔴", "alta": "🟠", "media": "🟡", "baixa": "🟢"}


def triar_na_demo(alerta: str, contexto: str):
    if not alerta.strip():
        raise gr.Error("Cole o alerta antes de triar.")
    inicio = time.perf_counter()
    saida, triagem = triar(alerta, contexto)
    tempo = time.perf_counter() - inicio

    if triagem is None:
        resumo = f"⚠️ **Saída inválida** ({tempo:.1f} s): o JSON não passou na validação. Encaminhar para triagem manual."
    else:
        resumo = (f"{ICONES[triagem['severidade']]} **Severidade {triagem['severidade']}** · "
                  f"{triagem['veredito'].replace('_', ' ')} · {tempo:.1f} s")
    return resumo, triagem, saida


exemplos = [
    [ex["alerta"], ex["contexto"]]
    for ex in [next(ex for ex in teste if ex["triagem"]["severidade"] == s) for s in ["critica", "alta", "media"]]
    + [next(ex for ex in teste if ex["triagem"]["veredito"] == "falso_positivo")]
]

with gr.Blocks(title="Triagem de alertas SOC") as demo:
    gr.Markdown(
        "# Assistente de triagem de alertas\n"
        f"Modelo `{MODEL_ID}` com adaptador LoRA ajustado para a política de severidade da organização."
    )
    with gr.Row():
        with gr.Column():
            alerta = gr.Textbox(label="Alerta (SIEM)", lines=3, placeholder="EDR | Host: ... | Usuário: ... | Processo: ...")
            contexto = gr.Textbox(label="Contexto (enriquecimento)", lines=3, placeholder="Ativo: ... | Usuário: ... | Reputação: ... | Ocorrências em 7 dias: ...")
            botao = gr.Button("Triar", variant="primary")
        with gr.Column():
            resumo = gr.Markdown()
            triagem = gr.JSON(label="Triagem")
            with gr.Accordion("Saída bruta do modelo", open=False):
                bruta = gr.Textbox(show_label=False, lines=4)
    gr.Examples(exemplos, inputs=[alerta, contexto], label="Alertas do conjunto de teste")
    botao.click(triar_na_demo, inputs=[alerta, contexto], outputs=[resumo, triagem, bruta])

### 7.4.2 Abrir a demo

Abrir a demo dentro do próprio Colab, logo abaixo da célula:

In [11]:
demo.launch(share=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

- No Colab, o Gradio cria por padrão um **link público** (`share=True`), válido por 72 horas, que qualquer pessoa com o endereço consegue abrir. Aqui ele fica desligado: os alertas têm hosts, usuários e IPs internos, e o argumento de **privacidade** do projeto é justamente que eles não saiam do ambiente.
- Para mostrar a demo a alguém fora do Colab, trocar para `share=True` e usar **só os exemplos do teste**, que são sintéticos. Nunca colar alertas reais numa demo com link público.
- Testar além dos exemplos: trocar a criticidade do ativo no contexto de um alerta (ex.: de `controlador de domínio` para `laboratório`) e ver se a severidade muda como a política manda. Esse é o comportamento que o fine-tuning deveria ter ensinado.

### 7.4.3 Fechar a demo

Ao terminar, fechar a demo para liberar a porta e a GPU:

In [ ]:
demo.close()

## 7.5 Limites da demo

A demo mostra o **núcleo** do assistente de SOC: a etapa de triagem. Num ambiente real, ainda faltaria:

| Limite | Na demo | Num SOC real |
|---|---|---|
| Entrada | Alerta e contexto colados à mão | O SIEM envia o alerta já enriquecido pela API |
| Saída inválida | Mostrada com aviso para triagem manual | Vai para a fila manual, com registro para a próxima iteração |
| Decisão final | O analista lê a triagem | O analista confirma ou corrige, e as correções viram dados de treino da próxima versão |
| Ambiente | Colab com GPU T4, enquanto o runtime estiver ativo | Servidor on-premises com GPU, sem acesso externo |
| Política | A que o modelo aprendeu no treino | Se a política mudar, é preciso novos dados e um novo treino (Fases 2 a 6) |

O último limite é o mais importante: a política fica **nos pesos** do adaptador. Mudar uma regra de severidade não é editar um prompt, é refazer o ciclo de fine-tuning. Esse é o custo da consistência que o projeto buscou.

## 7.6 Conclusão

Resultado da demo:

| Verificação | Resultado esperado | Onde conferir |
|---|---|---|
| Adaptador | O da Fase 5, medido na Fase 6, carregado do Drive num runtime novo | Seção 7.2.1 |
| Prompt | O mesmo `SYSTEM` do treino, lido do dataset da Fase 3 | Seção 7.2.2 |
| Triagem | JSON válido em alertas de cada severidade, em poucos segundos | Seção 7.3.2 |
| Demo | Alerta e contexto colados, triagem em JSON com resumo da severidade | Seção 7.4 |
| Privacidade | Demo sem link público, só com dados sintéticos | Seção 7.4.2 |

**Conclusão:** o adaptador LoRA treinado na Fase 5 e medido na Fase 6 virou um assistente de triagem que o analista L1 consegue usar: cola o alerta enriquecido e recebe o veredito, a severidade e a justificativa no formato que a fila do SOC e o SOAR consomem. O modelo roda numa GPU de entrada, com o mesmo prompt e a mesma validação da avaliação, então o comportamento da demo é o mesmo que os números da Fase 6 descrevem.